This notebook is dedicated to EDA (exploratory data analysis). It is meant to find hidden relationships before we start handling the main problem and solution.

In [1]:
#Adding Imports and setup

import os, sys, json, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from dateutil import parser as dateparser
from sklearn.model_selection import train_test_split

In [2]:
# Make plots a bit nicer
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

In [3]:
#Create Paths
project_root = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
data_dir = project_root / "data"

true_data_path = data_dir / "True.csv"
fake_data_path = data_dir / "Fake.csv"

true_data_path, fake_data_path, project_root


(PosixPath('/Users/karinahernandez/accenture-ai-studio-fall2025/data/True.csv'),
 PosixPath('/Users/karinahernandez/accenture-ai-studio-fall2025/data/Fake.csv'),
 PosixPath('/Users/karinahernandez/accenture-ai-studio-fall2025'))

In [26]:
#Load datasets
true_df = pd.read_csv(true_data_path)
fake_df = pd.read_csv(fake_data_path)

print("true shape:", true_df.shape)
print("fake shape:", fake_df.shape)

display(true_df.head(3))
display(fake_df.head(3))


true shape: (21417, 4)
fake shape: (23481, 4)


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"


Slightly more Fake articles than True ones.

In [35]:
#Adding labels and combining datasets
true_df = true_df.copy()
true_df["target"] = 1
true_df["source_tag"] = "true_csv"

fake_df = fake_df.copy()
fake_df["target"] = 0
fake_df["source_tag"] = "fake_csv"

df = pd.concat([true_df, fake_df], ignore_index=True)
print("combined shape:", df.shape)
df.sample(3, random_state=58)

combined shape: (44898, 6)


,title,text,subject,date,target,source_tag
3905,"With Obamacare vote, House Republicans free to...",WASHINGTON (Reuters) - The Republican-controll...,politicsNews,"May 5, 2017",1,true_csv
23776,The Vacationer-In-Chief Was Asked To Grade Hi...,Amateur president Donald Trump s first thirty ...,News,"February 28, 2017",0,fake_csv
11113,North Carolina's voter ID law goes on trial,"WINSTON-SALEM, N.C. (Reuters) - Minority voter...",politicsNews,"January 25, 2016",1,true_csv


Datasets are now combined into 1 huge data set with a new column known as "target" that refers to what is True or Fake.

In [36]:
# Missingness
missing = df.isna().mean().sort_values(ascending=False)
display(missing.head(20))


title         0.0
text          0.0
subject       0.0
date          0.0
target        0.0
source_tag    0.0
dtype: float64

No missing values found.

In [37]:
#Parsing dates to find potential patterns
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["dayofweek"] = df["date"].dt.dayofweek

In [38]:
df.head()

,title,text,subject,date,target,source_tag,year,month,dayofweek
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,2017-12-31,1,true_csv,2017.0,12.0,6.0
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,2017-12-29,1,true_csv,2017.0,12.0,4.0
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,2017-12-31,1,true_csv,2017.0,12.0,6.0
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,2017-12-30,1,true_csv,2017.0,12.0,5.0
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,2017-12-29,1,true_csv,2017.0,12.0,4.0


In [39]:
#Quick text features for EDA: title and text lengths and words
df["title_len"] = df["title"].astype(str).str.len()
df["text_len"]  = df["text"].astype(str).str.len()
df["title_words"] = df["title"].astype(str).str.split().str.len()
df["text_words"]  = df["text"].astype(str).str.split().str.len()


In [40]:
df.head()

,title,text,subject,date,target,source_tag,year,month,dayofweek,title_len,text_len,title_words,text_words
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,2017-12-31,1,true_csv,2017.0,12.0,6.0,64,4659,10,749
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,2017-12-29,1,true_csv,2017.0,12.0,4.0,64,4077,9,624
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,2017-12-31,1,true_csv,2017.0,12.0,6.0,60,2789,10,457
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,2017-12-30,1,true_csv,2017.0,12.0,5.0,59,2461,9,376
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,2017-12-29,1,true_csv,2017.0,12.0,4.0,69,5204,11,852
